<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/movie_recommendation_system_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature engineering

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
DATASET_PATH = "/content/drive/MyDrive/ml-32m/ml-32m"

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

import warnings
warnings.filterwarnings("ignore")

In [3]:
movies = pd.read_csv(DATASET_PATH + "/" + "movies.csv")
ratings = pd.read_csv(DATASET_PATH + "/" + "ratings.csv")
tags = pd.read_csv(DATASET_PATH + "/" + "tags.csv")
links = pd.read_csv(DATASET_PATH + "/" + "links.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ml-32m/ml-32m/movies.csv'

In [ ]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

In [ ]:
movies["clean_title"] = (
    movies["title"]
    .str.replace(r"\(\d{4}\)", "", regex=True)
    .str.strip()
)

In [ ]:
movies["genres_list"] = movies["genres"].str.split("|")

In [ ]:
ratings["timestamp"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s"
)

In [ ]:
tags["timestamp"] = pd.to_datetime(
    tags["timestamp"],
    unit="s"
)

In [ ]:
tags = tags.dropna(subset=["tag"])

In [ ]:
movie_data = movies.merge(
    links,
    on="movieId",
    how="left"
)

In [ ]:
movie_data.head()

In [ ]:
movie_tags = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

In [ ]:
movie_tags.head()

In [ ]:
movie_data = movie_data.merge(
    movie_tags,
    on="movieId",
    how="left"
)

In [ ]:
movie_data["tag"] = movie_data["tag"].fillna("")

In [ ]:
movie_data["combined_features"] = (
    movie_data["clean_title"]
    + " "
    + movie_data["genres"].str.replace("|", " ", regex=False)
    + " "
    + movie_data["tag"]
)

In [ ]:
movie_data[
    [
        "clean_title",
        "genres",
        "tag",
        "combined_features"
    ]
].head()

In [ ]:
movie_data["tag"] = movie_data["tag"].str.lower()

In [ ]:
import re

movie_data["tag"] = movie_data["tag"].apply(
    lambda x: re.sub(r"[^a-zA-Z0-9\s]", " ", x)
)

In [ ]:
movie_data["tag"] = movie_data["tag"].str.replace(r"\s+", " ", regex=True).str.strip()

In [ ]:
import re

movie_data["tag"] = (
    movie_data["tag"]
    .str.lower()
    .str.encode("ascii", "ignore")
    .str.decode("ascii")
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

In [ ]:
tfidf_matrix = tfidf.fit_transform(
    movie_data["combined_features"]
)

In [ ]:
print(tfidf_matrix.shape)

In [ ]:
movie_stats = (
    ratings.groupby("movieId")
    .agg(
        average_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

In [ ]:
movie_stats = movie_stats.merge(
    movie_data,
    on="movieId",
    how="left"
)

In [ ]:
movie_stats.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from scipy.sparse import save_npz
import pickle

In [ ]:
user_encoder = LabelEncoder()
ratings["user_idx"] = user_encoder.fit_transform(ratings["userId"])

In [ ]:
movie_encoder = LabelEncoder()
ratings["movie_idx"] = movie_encoder.fit_transform(ratings["movieId"])

In [ ]:
user_item_sparse = csr_matrix(
    (
        ratings["rating"],
        (
            ratings["user_idx"],
            ratings["movie_idx"]
        )
    )
)

In [ ]:
print(user_item_sparse.shape)

In [ ]:
save_npz(
    "user_item_sparse.npz",
    user_item_sparse
)

In [ ]:
with open("user_encoder.pkl", "wb") as f:
    pickle.dump(user_encoder, f)

with open("movie_encoder.pkl", "wb") as f:
    pickle.dump(movie_encoder, f)

In [ ]:
import pickle

In [ ]:
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

In [ ]:
with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

In [ ]:
movie_data.to_pickle("movie_data.pkl")

In [ ]:
movie_stats.to_pickle("movie_stats.pkl")